# API Integration Patterns

Learn best practices for integrating with the ADEPT API ecosystem.

## Security Architecture

All API requests route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net`.

## Topics Covered
- OpenAI-compatible API
- Response API vs Assistant API
- WebSocket connections
- Error handling
- Rate limiting

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
import os
import json
import httpx
import asyncio
from typing import AsyncIterator

# Load config
config = json.load(open("/tmp/adept_config.json"))
auth = json.load(open("/tmp/adept_auth.json"))

API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))

async def stream_chat_response(prompt: str) -> AsyncIterator[str]:
    '''Stream responses from the agent via nginx_proxy.'''
    
    async with httpx.AsyncClient(verify=False, timeout=60.0) as client:
        headers = {"Authorization": f"Bearer {auth['access_token']}"}
        
        async with client.stream(
            "POST",
            f"{API_BASE_URL}/chat/completions",
            json={
                "model": "agentic-framework:scientific-agent-v1",
                "messages": [{"role": "user", "content": prompt}],
                "stream": True
            },
            headers=headers
        ) as response:
            async for line in response.aiter_lines():
                if line.startswith("data: "):
                    yield line[6:]

print("Streaming client configured")
print(f"   API: {API_BASE_URL}")
print(f"   Routing: nginx_proxy (TLS + JWT)")